# 01 – Dutch Retail Sales Index EDA
**Stakeholder:** Dutch Retail Category Manager


In [1]:
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/dutch_retail_sales_index.csv', index_col=0)
df.columns = ['time_decimal', 'sales_index']
df['year'] = df['time_decimal'].astype(int)
df['month'] = ((df['time_decimal'] - df['year']) * 12).round().astype(int) + 1
df['month'] = df['month'].clip(1, 12)
df['date'] = pd.to_datetime(df[['year','month']].assign(day=1))
df = df.sort_values('date').reset_index(drop=True)
print('Shape:', df.shape)
print('Date range:', df['date'].min(), '->', df['date'].max())
print('Missing:', df.isnull().sum().sum())
print(df[['sales_index']].describe().round(1))


Shape: (425, 5)
Date range: 1960-05-01 00:00:00 -> 1995-09-01 00:00:00
Missing: 0
       sales_index
count        425.0
mean          63.4
std           32.2
min           12.0
25%           30.0
50%           73.0
75%           88.0
max          131.0


In [2]:
fig,ax=plt.subplots(figsize=(14,5))
ax.plot(df['date'],df['sales_index'],alpha=0.4,color='steelblue',label='Monthly')
ax.plot(df['date'],df['sales_index'].rolling(12).mean(),color='darkblue',lw=2,label='12m avg')
ax.set_title('Dutch Retail Sales Index – Full Trend')
ax.legend()
plt.tight_layout()
plt.savefig('../data/q1_trend.png',dpi=150)
print('Saved q1_trend.png')


Saved q1_trend.png


In [3]:
seasonal=df.groupby('month')['sales_index'].mean()
months=['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
fig,ax=plt.subplots(figsize=(10,4))
ax.bar(months,seasonal.values,color='steelblue')
ax.set_title('Avg Sales Index by Month')
plt.tight_layout()
plt.savefig('../data/q2_seasonal.png',dpi=150)
print('Peak:', months[seasonal.values.argmax()], '| Trough:', months[seasonal.values.argmin()])


Peak: Dec | Trough: Feb


In [4]:
df['decade']=(df['year']//10)*10
print(df.groupby('decade')['sales_index'].mean().round(1))


decade
1960     21.8
1970     57.0
1980     85.6
1990    106.1
Name: sales_index, dtype: float64


In [5]:
annual=df.groupby('year')['sales_index'].mean()
yoy=annual.pct_change()*100
fig,(ax1,ax2)=plt.subplots(2,1,figsize=(12,7),sharex=True)
ax1.bar(yoy.index,yoy,color=['green' if v>0 else 'red' for v in yoy])
ax1.set_title('YoY % Change')
ax1.axhline(0,color='black',lw=0.8)
ax2.plot(yoy.rolling(10).std(),color='purple')
ax2.set_title('10-Year Rolling Volatility')
plt.tight_layout()
plt.savefig('../data/q5_yoy.png',dpi=150)
print('Saved q5_yoy.png')


Saved q5_yoy.png
